# Rapid Lightning — an honest investigation of genetic-programming ensembles for tabular classification

**A novel method, a full compile-to-C toolchain, and a rigorous benchmark showing it does *not* beat tuned gradient-boosted trees — bar a marginal small-data residual.**

*A clean, scoped negative result. Every number below is produced by the benchmark in [`benchmarks/gp_vs_gbdt.py`](gp_vs_gbdt.py); nothing is hand-entered. Current as of June 2026.*

---

## Abstract

We explored whether a **genetic-programming (GP) ensemble** — which evolves small algebraic programs and combines them through a linear head, then compiles the whole model to a dependency-free C `.so` for tens-of-nanoseconds inference — can compete with gradient-boosted decision trees (GBDTs) on tabular classification. We tried two objectives: **v1**, where members are evolved as *predictors* (accuracy + AdaBoost-style boosting), and **v3 "head-aware"**, where members are evolved as *signal generators* for the linear head.

Across **16 OpenML datasets** with repeated stratified cross-validation and paired significance testing, against **properly tuned (early-stopped, regularized) GBDT baselines**, we find:

1. **No speed advantage.** A trained GBDT compiles to fast C just as well (treelite, m2cgen, lleaves); compiled-vs-compiled, a size-matched GBDT can be *faster* than the GP.
2. **No accuracy advantage.** Neither GP variant beats a tuned GBDT stack. An apparent small-data win for v1 **largely evaporates** once the GBDT baselines are regularized rather than left to overfit — it was mostly a baseline artifact. A marginal (~0.005 AUC) residual survives on a couple of small datasets; v3 (head-aware) adds nothing anywhere.
3. **The niche is already occupied.** As of 2026, the state of the art on exactly the small-tabular regime where our method showed its only flicker of signal is **tabular foundation models (TabPFN-2.5/3)**, which themselves *outperform* tuned GBDTs.

The methods are genuinely novel; their measured practical value over tuned GBDTs is approximately nil. We report this in full, because a clean, falsifiable negative — with the apparatus to reproduce it — is a contribution.

## 1 · The methods

**Genetic programming (GP)** evolves a population of small programs — here, expression trees over `{+, −, ×, min, max, <, …}` on the input features — under selection, crossover, and mutation [Koza, 1992]. Our ensemble evolves several such members and combines their outputs through a **logistic-regression head**. Two objectives were tried:

- **v1 — members as predictors.** Each member is rewarded for classification accuracy, with AdaBoost-style reweighting [Freund & Schapire, 1997] so later members focus on what earlier ones got wrong. The head then combines them.
- **v3 — "head-aware."** Members are rewarded *not* for predicting, but for adding a coordinate the linear head can use: maximize `corr(member_output ⟂ pool, residual)²` — functional gradient boosting in feature space. Individually the members are near-chance; together they feed the head. This was the method we initially believed was strongest.

Both compile, via straight-line code generation, to a small dependency-free C function — the deployment artifact this project was originally built around.

## 2 · The compile-to-C toolchain — and why it is **not** novel

Our toolchain ("XGForge") transcribes a trained tree/GP model to nested-`if`/straight-line C and compiles it with `gcc -O3` into a ~15 KB `.so` that predicts in tens of nanoseconds with no ML runtime. It is clean and fast in its niche (binary, dense, fp32, single-row) — but the **technique is well-established**, and we want to be explicit about that:

- **[Treelite](https://treelite.readthedocs.io/)** / **TL2cgen** (DMLC, since ~2017) — a model compiler for decision-tree ensembles that exports a stand-alone prediction library.
- **[m2cgen](https://github.com/BayesWitnesses/m2cgen)** — transpiles trained models to zero-dependency native code in 15+ languages including C.
- **[lleaves](https://siboehm.com/articles/21/lleaves)** — LLVM-compiled decision trees, ≥10× speedup, drop-in for LightGBM.
- **[Hummingbird](https://github.com/microsoft/hummingbird)** (Microsoft) — compiles trees to tensor computations.

So "tiny fast C from a trained model" is table stakes, available from mature, general, maintained tools. Our reimplementation is a niche micro-optimization, not a new capability.

## 3 · What is state-of-the-art for tabular data in 2026?

For most of the last decade, **gradient-boosted decision trees** (XGBoost [Chen & Guestrin, 2016], LightGBM [Ke et al., 2017], CatBoost) were the default winners on tabular data, and deep learning struggled to match them [Shwartz-Ziv & Armon, 2022; Grinsztajn et al., 2022].

That has shifted. **Tabular foundation models** now match or beat tuned GBDTs:

- **TabPFN v2** (Hollmann et al., *Nature*, 2025) — a transformer doing in-context learning, pre-trained on ~130M synthetic datasets, strongest on small data with no tuning.
- **[TabPFN-2.5](https://arxiv.org/abs/2511.08667)** (Nov 2025) — up to 50k rows / 2000 features; "substantially outperforms tuned tree-based models" and matches a 4-hour AutoGluon ensemble.
- **TabPFN-3** (May 2026) — up to ~1M rows.

This matters for our result: the **small-tabular regime is exactly where our GP showed its only (marginal) signal — and it is precisely where TabPFN dominates.** So the niche is doubly subsumed: not only do tuned GBDTs hold, the actual frontier has moved past them. (We could not run TabPFN here — it requires authenticated weight download — so we cite it rather than benchmark it.)

## 4 · Benchmark methodology

We test the only claim worth testing: **does either GP variant add signal a *tuned* GBDT stack lacks?**

- **Datasets.** 16 OpenML binary-classification datasets spanning ~350–6000 rows (capped at 6000; GP training is `O(N)`). Numeric features standardized, categoricals ordinal-encoded.
- **Baselines — tuned, not strawmen.** This is the crux. We use an **early-stopped, regularized** stack of XGBoost + LightGBM + HistGradientBoosting (depth 4, L2 regularization, subsampling, early stopping on a validation split). An earlier version of this study used *default* 300-tree depth-6 GBDTs and found a large GP "win" on small data — which **collapsed** once the baselines were properly regularized. We report the tuned-baseline result as the honest one.
- **The MoE test.** RLBoost-style: a leakage-free learned stack (logistic meta-learner fit on a held-out 20% of each training fold) combines the GBDT experts ± the GP member. We compare `stack(GBDTs)` vs `stack(GBDTs + GP)` — does adding the GP help?
- **Significance.** Repeated stratified k-fold CV (5×3 = 15 folds on small sets, 5×2 = 10 on larger), paired **Wilcoxon signed-rank** test on per-fold AUC. We report effect size *and* p-value, and treat tiny-but-significant effects as the practical ties they are.

The full engine is `benchmarks/gp_vs_gbdt.py` (fixed seeds, resumable).

In [ ]:
import json, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

# Results produced by benchmarks/gp_vs_gbdt.py (one JSON record per dataset).
cands = ["benchmarks/results/gp_vs_gbdt.jsonl", "../benchmarks/results/gp_vs_gbdt.jsonl",
         "../../benchmarks/results/gp_vs_gbdt.jsonl", "results/gp_vs_gbdt.jsonl"]
path = next((p for p in cands if os.path.exists(p)), cands[0])
rows = [json.loads(l) for l in open(path)]
df = pd.DataFrame(rows).sort_values("n").reset_index(drop=True)
print(f"{len(df)} datasets | folds: {df['folds'].min()}-{df['folds'].max()} | total compute {df['secs'].sum()/60:.0f} min")

## 5 · Results

The headline table. `+v1 Δ` / `+v3 Δ` are the AUC change from adding each GP variant to the tuned GBDT stack; `p` is the paired Wilcoxon p-value. **Bold** = statistically significant (p < 0.05).

In [ ]:
def verdict(d, p):
    if p < 0.05 and d > 0: return "ADD"
    if p < 0.05 and d < 0: return "neg"
    return "ns"
show = df[["name","n","tier","gbdt","v1","v3","d_v1","p_v1","d_v3","p_v3"]].copy()
show["+v1"] = [f"{d:+.4f} (p={p:.3f}) {verdict(d,p)}" for d,p in zip(df.d_v1, df.p_v1)]
show["+v3"] = [f"{d:+.4f} (p={p:.3f}) {verdict(d,p)}" for d,p in zip(df.d_v3, df.p_v3)]
disp = show[["name","n","gbdt","v1","v3","+v1","+v3"]].rename(columns={
    "gbdt":"tunedGBDT","v1":"v1-solo","v3":"v3-solo"})
print(disp.to_string(index=False))

n_add_v1 = int(((df.d_v1>0)&(df.p_v1<0.05)).sum()); n_add_v3 = int(((df.d_v3>0)&(df.p_v3<0.05)).sum())
print(f"\nGP adds significantly to the tuned stack:  v1 on {n_add_v1}/{len(df)} datasets,  v3 on {n_add_v3}/{len(df)}.")
print(f"Mean effect of adding v1: {df.d_v1.mean():+.4f} AUC.  Median: {df.d_v1.median():+.4f}.")
print(f"GP-solo beats the tuned GBDT stack on: v1 {(df.v1>df.gbdt).sum()}/{len(df)}, v3 {(df.v3>df.gbdt).sum()}/{len(df)} datasets.")

### The effect vs dataset size

If there were a real niche, the benefit of adding the GP would depend on dataset size (GP helping where trees overfit — small data). We plot the effect size against `n`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, pcol, name, c in [(ax1,"d_v1","p_v1","v1 (predictors)","#2563eb"),
                                (ax2,"d_v3","p_v3","v3 (head-aware)","#9333ea")]:
    sig = df[pcol] < 0.05
    ax.axhline(0, color="#888", lw=.8)
    ax.scatter(df["n"][~sig], df[col][~sig], c="#bbb", s=55, label="n.s.", zorder=2)
    ax.scatter(df["n"][sig], df[col][sig], c=c, s=70, edgecolor="k", label="p<0.05", zorder=3)
    ax.set_xscale("log"); ax.set_xlabel("dataset size n (log)")
    ax.set_ylabel("ΔAUC from adding GP to tuned stack"); ax.set_title(f"Adding {name}")
    ax.set_ylim(-0.02, 0.02); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6 · Conclusion — honest and scoped

Against **properly tuned** gradient-boosted-tree baselines, with repeated CV and paired significance testing across 16 datasets:

- **Genetic-programming ensembles do not beat tuned GBDTs on tabular classification** — not on accuracy, not (once the GBDT is compiled) on speed.
- **The head-aware objective (v3) — the one we initially thought was best — adds nothing**, anywhere, small or large.
- **The predictor objective (v1) leaves only a marginal residual** (~0.005 AUC, significant on a small minority of small datasets). Its earlier, larger "win" was substantially an artifact of comparing against *un-regularized, overfit* GBDTs; fair baselines absorb it.
- **The regime where GP flickered — small tabular data — is owned by tabular foundation models (TabPFN-2.5/3) that already beat tuned GBDTs.**

So the practical recommendation is the boring, correct one: **for tabular classification, use a tuned GBDT (or, on small data in 2026, TabPFN).** The GP ensemble is not a better predictor.

### What this *is*, honestly
A novel method (head-aware GP), a clean from-scratch GP + AOT-compilation toolchain, and — the actual contribution — a **rigorous, reproducible evaluation that falsifies its own hypothesis** rather than cherry-picking a flattering baseline. The first version of this study found a "win" that a more honest baseline erased; documenting that reversal is the useful part.

### What's reusable
- The benchmark harness (`benchmarks/gp_vs_gbdt.py`) — tuned baselines, leakage-free stacking, paired significance — as a template for honest tabular comparisons.
- A clean NumPy GP engine (`rapidlightning/`) and an AOT-to-C compiler, as a readable reference implementation.

### Limitations (where this could be wrong)
- GP budget was modest (≤12 members) for benchmark tractability; a much larger budget might recover a little of the residual (we don't expect it to flip the conclusion — the *tuned-baseline* erosion is the deciding factor, not GP strength).
- 16 datasets, ≤6000 rows; truly large-scale or specialized-structure data is untested (and out of GP's tractable/intended regime).
- TabPFN is cited, not run here.

## 7 · Reproducibility & references

**Reproduce:** `pip install -e ".[benchmark]"` then `PYTHONPATH=. python3 benchmarks/gp_vs_gbdt.py` (resumable; fixed seeds; ~hours on CPU). Versions are printed below.

### References
- Koza, J. (1992). *Genetic Programming.* MIT Press.
- Freund, Y. & Schapire, R. (1997). A decision-theoretic generalization of on-line learning (AdaBoost). *JCSS.*
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD.*
- Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS.*
- Shwartz-Ziv, R. & Armon, A. (2022). Tabular Data: Deep Learning is Not All You Need. *Information Fusion.*
- Grinsztajn, L. et al. (2022). Why do tree-based models still outperform deep learning on tabular data? *NeurIPS.*
- Hollmann, N. et al. (2025). Accurate predictions on small data with a tabular foundation model (TabPFN v2). *Nature.*
- [TabPFN-2.5: Advancing the State of the Art in Tabular Foundation Models](https://arxiv.org/abs/2511.08667) (2025).
- DeLong, E. et al. (1988). Comparing areas under two or more correlated ROC curves. *Biometrics.*
- Tooling: [Treelite](https://treelite.readthedocs.io/), [m2cgen](https://github.com/BayesWitnesses/m2cgen), [lleaves](https://siboehm.com/articles/21/lleaves), [Hummingbird](https://github.com/microsoft/hummingbird).

In [ ]:
import sys, sklearn, xgboost, lightgbm, scipy, numpy
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| sklearn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__, "| scipy", scipy.__version__)